# Notebook 03 — Sinh caption (cấu hình A, B, C, D)

| Cấu hình | Mô hình | Ảnh | Cách làm |
|---|---|---|---|
| **A** gốc | LLaVA-OneVision-7B, BLIP-2, Qwen3-VL-8B | Toàn bộ | Chỉ dẫn cố định |
| **B** ngữ cảnh | LLaVA-OneVision-7B | SciCap+ | Thêm đoạn văn nhắc đến hình vào chỉ dẫn |
| **C** tự kiểm tra | LLaVA-OneVision-7B | Biểu đồ, SciCap+, 30 ảnh COCO | Từ caption A: tách ý → hỏi lại từng ý → viết lại nếu có ý sai |
| **D** cả hai | LLaVA-OneVision-7B | SciCap+ | B rồi C |

**Cách chạy:** Runtime → T4 GPU. Chạy mục 1–2, rồi **chỉ một** trong các mục 3, 4, 5 cho mỗi phiên (mỗi mô hình một phiên để không hết bộ nhớ). Kết quả lưu sau từng ảnh vào Drive; nếu Colab ngắt, chạy lại đúng ô đó sẽ tiếp tục từ ảnh chưa làm.

## 1. Cài đặt, Drive, dữ liệu

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import gc, json, os, re, time
# Giảm phân mảnh bộ nhớ GPU, theo gợi ý trong thông báo lỗi của PyTorch; phải đặt trước khi dùng CUDA
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
import transformers
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/ImageCaptioning"
DATA_DIR, PRED_DIR = f"{BASE}/data", f"{BASE}/ketqua"
os.makedirs(PRED_DIR, exist_ok=True)
with open(f"{DATA_DIR}/dataset.json", encoding="utf8") as f:
    dataset = json.load(f)
ITEMS = dataset["items"]
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có (chỉ chạy được mục 6)"
print(dataset["meta"]["counts"], "| transformers", transformers.__version__, "| GPU", gpu)

PROMPT_A = "Describe this image in one short sentence."
MAX_NEW_TOKENS = 60
N_COCO_FOR_C = 30  # cấu hình C tốn nhiều lần gọi mô hình nên chỉ chạy trên tập con COCO

## 2. Hàm dùng chung

In [ ]:
QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)


def load(model_id):
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(model_id, quantization_config=QUANT, device_map="auto")
    return model, processor


DOWNSCALE = None  # chỉ đặt khi một ảnh bị hết bộ nhớ GPU (xem run_config)


def image_of(item):
    image = Image.open(os.path.join(DATA_DIR, item["file"])).convert("RGB")
    if DOWNSCALE:
        image.thumbnail((DOWNSCALE, DOWNSCALE))
    return image


def chat(model, processor, text, image=None, max_new_tokens=MAX_NEW_TOKENS):
    content = ([{"type": "image", "image": image}] if image is not None else []) + [{"type": "text", "text": text}]
    inputs = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors="pt")
    inputs = inputs.to(model.device, torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()


def run_config(model_name, config, items, caption_fn, prompt_note):
    """Chạy một cấu hình, lưu sau từng ảnh, bỏ qua ảnh đã làm nếu chạy lại."""
    path = f"{PRED_DIR}/captions_{model_name}_{config}.json"
    if os.path.exists(path):
        with open(path, encoding="utf8") as f:
            run = json.load(f)
    else:
        run = {"model": model_name, "config": config, "prompt": prompt_note, "quantization": "4-bit nf4",
               "transformers": transformers.__version__, "captions": {}}
    todo = [it for it in items if it["id"] not in run["captions"]]
    print(f"{model_name} {config}: {len(run['captions'])} đã có, còn {len(todo)}")
    torch.cuda.reset_peak_memory_stats()
    global DOWNSCALE
    for n, item in enumerate(todo, 1):
        start = time.time()
        try:
            result = caption_fn(item)
            downscaled = None
        except torch.cuda.OutOfMemoryError:
            # Ảnh quá lớn → AnyRes sinh quá nhiều token ảnh. Thử lại riêng ảnh này sau khi thu nhỏ.
            gc.collect(); torch.cuda.empty_cache()
            DOWNSCALE = 768
            try:
                result = caption_fn(item)
            finally:
                DOWNSCALE = None
            downscaled = 768
            print(f"  [{item['id']}] hết bộ nhớ GPU ở độ phân giải gốc → thu nhỏ còn {downscaled}px")
        if isinstance(result, str):
            result = {"text": result}
        result["seconds"] = round(time.time() - start, 1)
        if downscaled:
            result["downscaled_to"] = downscaled
        torch.cuda.empty_cache()
        run["captions"][item["id"]] = result
        run["gpu_gb_peak"] = round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2)
        with open(path, "w", encoding="utf8") as f:
            json.dump(run, f, ensure_ascii=False, indent=1)
        if n % 10 == 0 or n == len(todo):
            print(f"  {n}/{len(todo)}  [{item['id']}] {result['text'][:90]}")
    return run

## 3. LLaVA-OneVision-7B: cấu hình A, B, C, D

Thứ tự trong một phiên: A trước (C và D dùng lại kết quả của A và B).

In [ ]:
LLAVA = "LLaVA-OneVision-7B"
model, processor = load("llava-hf/llava-onevision-qwen2-7b-ov-hf")

# ---- A: chỉ dẫn cố định cho mọi ảnh ----
run_A = run_config(LLAVA, "A", ITEMS, lambda it: chat(model, processor, PROMPT_A, image_of(it)), PROMPT_A)

In [ ]:
# ---- B: thêm đoạn văn nhắc đến hình (chỉ SciCap+) ----
def prompt_B(item, max_chars=1200):
    context = " ".join(item["mention_paragraphs"])[:max_chars]
    return ("Here is text from the paper that refers to this figure:\n"
            f"\"{context}\"\n"
            "Using the figure and this text, describe the figure in one short sentence.")


SCICAP = [it for it in ITEMS if it["source"] == "scicap_plus_test"]
run_B = run_config(LLAVA, "B", SCICAP, lambda it: chat(model, processor, prompt_B(it), image_of(it)),
                   "Đoạn văn nhắc đến hình (tối đa 1200 ký tự) + yêu cầu mô tả một câu")

In [ ]:
# ---- C: tự kiểm tra (không dùng mô hình bên ngoài, khác Woodpecker [42]) ----
def extract_claims(caption):
    # Bước 1: tách caption thành các ý ngắn (chỉ dùng phần ngôn ngữ, không cần ảnh)
    text = chat(model, processor,
                "Split the following caption into short, separate factual statements, one per line. "
                "Include every object, attribute, number and trend it mentions. Caption: " + caption,
                max_new_tokens=120)
    claims = [re.sub(r"^[\s\-\*\d\.\)]+", "", line).strip() for line in text.splitlines()]
    return [c for c in claims if len(c.split()) >= 2][:6]


def verify(claim, image):
    # Bước 2: hỏi lại mô hình, có nhìn ảnh, từng ý đúng hay sai
    answer = chat(model, processor, f"Is the following statement true for this image? Answer yes or no.\nStatement: {claim}",
                  image, max_new_tokens=3)
    return answer.lower().startswith("yes")


def self_check(item, caption):
    image = image_of(item)
    claims = extract_claims(caption)
    checks = [(c, verify(c, image)) for c in claims]
    wrong = [c for c, ok in checks if not ok]
    final = caption
    if wrong:
        # Bước 3: viết lại, bỏ các ý bị mô hình cho là sai
        final = chat(model, processor,
                     f"Rewrite this caption as one short sentence, removing the incorrect statements.\n"
                     f"Caption: {caption}\nIncorrect statements: " + "; ".join(wrong), image)
    return {"text": final, "original": caption, "claims": [{"claim": c, "supported": ok} for c, ok in checks]}


C_ITEMS = ([it for it in ITEMS if it["source"] in ("synthetic_chart", "scicap_plus_test")]
           + [it for it in ITEMS if it["source"] == "coco_val2017"][:N_COCO_FOR_C])
run_C = run_config(LLAVA, "C", C_ITEMS, lambda it: self_check(it, run_A["captions"][it["id"]]["text"]),
                   "Caption A → tách ý → hỏi lại từng ý (có/không) → viết lại nếu có ý sai")

In [ ]:
# ---- D: ngữ cảnh rồi tự kiểm tra (chỉ SciCap+) ----
run_D = run_config(LLAVA, "D", SCICAP, lambda it: self_check(it, run_B["captions"][it["id"]]["text"]),
                   "Caption B → tự kiểm tra như C")
del model, processor
gc.collect(); torch.cuda.empty_cache()

## 4. BLIP-2 OPT 2.7B: cấu hình A (phiên riêng)

In [ ]:
model, processor = load("Salesforce/blip2-opt-2.7b")


def blip2_caption(item):
    # BLIP-2 sinh caption không cần chỉ dẫn, theo cách dùng chuẩn của mô hình
    inputs = processor(images=image_of(item), return_tensors="pt").to(model.device, torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()


run_config("BLIP-2", "A", ITEMS, blip2_caption, "Không dùng chỉ dẫn (cách dùng chuẩn của BLIP-2)")
del model, processor
gc.collect(); torch.cuda.empty_cache()

## 5. Qwen3-VL-8B: cấu hình A (phiên riêng)

In [ ]:
model, processor = load("Qwen/Qwen3-VL-8B-Instruct")
run_config("Qwen3-VL-8B", "A", ITEMS, lambda it: chat(model, processor, PROMPT_A, image_of(it)), PROMPT_A)
del model, processor
gc.collect(); torch.cuda.empty_cache()

## 6. Kiểm tra các file kết quả

In [ ]:
import glob
print("Số ảnh cần có: A =", len(ITEMS), "| B, D =", sum(it["source"] == "scicap_plus_test" for it in ITEMS),
      "| C =", sum(it["source"] in ("synthetic_chart", "scicap_plus_test") for it in ITEMS) + N_COCO_FOR_C)
for path in sorted(glob.glob(f"{PRED_DIR}/captions_*.json")):
    with open(path, encoding="utf8") as f:
        run = json.load(f)
    if not run["captions"]:
        continue
    times = [c["seconds"] for c in run["captions"].values()]
    small = [i for i, c in run["captions"].items() if c.get("downscaled_to")]
    print(f"{os.path.basename(path):45s} {len(run['captions']):4d} ảnh | "
          f"{sum(times) / len(times):5.1f} s/ảnh | GPU cao nhất {run.get('gpu_gb_peak')} GB | "
          f"thu nhỏ do hết bộ nhớ: {len(small)} {small}")